# Hand Gesture Dataset EDA

数据集探索性分析：类别分布、Bbox 尺寸/位置分布、图像属性统计。

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import pandas as pd

plt.rcParams["figure.dpi"] = 120

DATA_ROOT = Path("../../datasets/RPS")
CLASS_NAMES = {0: "Stop", 1: "Understand", 2: "NumberTwo"}
SPLITS = ["train", "valid", "test"]

## 1. 数据集概览

In [ ]:
# Count images per split
split_counts = {}
for split in SPLITS:
    img_dir = DATA_ROOT / split / "images"
    if img_dir.exists():
        split_counts[split] = len(list(img_dir.glob("*")))
    else:
        split_counts[split] = 0

fig, ax = plt.subplots(figsize=(6, 3))
bars = ax.bar(split_counts.keys(), split_counts.values(), color=["#4C78A8", "#F58518", "#E45756"])
ax.bar_label(bars)
ax.set_ylabel("Number of images")
ax.set_title("Images per Split")
plt.tight_layout()
plt.show()
print(split_counts)

In [ ]:
# Show sample images with annotations
def draw_yolo_boxes(img, labels, class_names):
    h, w = img.shape[:2]
    colors = [(76, 120, 168), (245, 133, 24), (228, 87, 86)]
    for line in labels:
        parts = line.strip().split()
        if len(parts) != 5:
            continue
        cls_id = int(parts[0])
        cx, cy, bw, bh = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
        x1 = int((cx - bw/2) * w)
        y1 = int((cy - bh/2) * h)
        x2 = int((cx + bw/2) * w)
        y2 = int((cy + bh/2) * h)
        color = colors[cls_id % len(colors)]
        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
        cv2.putText(img, class_names.get(cls_id, str(cls_id)), (x1, y1 - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    return img

img_dir = DATA_ROOT / "train" / "images"
lbl_dir = DATA_ROOT / "train" / "labels"
sample_imgs = sorted(img_dir.glob("*"))[:9]

fig, axes = plt.subplots(3, 3, figsize=(12, 12))
for ax, img_path in zip(axes.flat, sample_imgs):
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    lbl_path = lbl_dir / (img_path.stem + ".txt")
    if lbl_path.exists():
        labels = lbl_path.read_text().strip().split("\n")
        img = draw_yolo_boxes(img, labels, CLASS_NAMES)
    ax.imshow(img)
    ax.set_title(img_path.name, fontsize=8)
    ax.axis("off")
plt.suptitle("Sample Training Images with Annotations", fontsize=14)
plt.tight_layout()
plt.show()

## 2. 标注分析

In [ ]:
# Parse all annotations into a DataFrame
records = []
for split in SPLITS:
    lbl_dir = DATA_ROOT / split / "labels"
    if not lbl_dir.exists():
        continue
    for lbl_file in lbl_dir.glob("*.txt"):
        for line in lbl_file.read_text().strip().split("\n"):
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            records.append({
                "split": split,
                "image": lbl_file.stem,
                "class_id": int(parts[0]),
                "cx": float(parts[1]),
                "cy": float(parts[2]),
                "w": float(parts[3]),
                "h": float(parts[4]),
            })

df = pd.DataFrame(records)
df["class_name"] = df["class_id"].map(CLASS_NAMES)
df["area"] = df["w"] * df["h"]
df["aspect_ratio"] = df["w"] / df["h"].clip(lower=1e-6)
print(f"Total annotations: {len(df)}")
df.head()

In [ ]:
# Class distribution per split
fig, axes = plt.subplots(1, len(SPLITS), figsize=(12, 4), sharey=True)
for ax, split in zip(axes, SPLITS):
    sub = df[df["split"] == split]
    counts = sub["class_name"].value_counts().reindex(CLASS_NAMES.values(), fill_value=0)
    bars = ax.bar(counts.index, counts.values, color=["#4C78A8", "#F58518", "#E45756"])
    ax.bar_label(bars)
    ax.set_title(f"{split} ({len(sub)} annotations)")
    ax.tick_params(axis="x", rotation=30)
plt.suptitle("Class Distribution per Split", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Bbox area distribution by class
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
for ax, (cls_id, cls_name) in zip(axes, CLASS_NAMES.items()):
    sub = df[df["class_id"] == cls_id]
    ax.hist(sub["area"], bins=50, alpha=0.8, edgecolor="white")
    ax.set_title(f"{cls_name} (n={len(sub)})")
    ax.set_xlabel("Relative Area (w*h)")
axes[0].set_ylabel("Count")
plt.suptitle("Bounding Box Area Distribution", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Bbox center heatmap
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (cls_id, cls_name) in zip(axes, CLASS_NAMES.items()):
    sub = df[df["class_id"] == cls_id]
    ax.hist2d(sub["cx"], sub["cy"], bins=30, cmap="YlOrRd")
    ax.set_title(cls_name)
    ax.set_xlabel("cx")
    ax.set_ylabel("cy")
    ax.invert_yaxis()
    ax.set_aspect("equal")
plt.suptitle("Bounding Box Center Distribution", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Aspect ratio distribution
fig, ax = plt.subplots(figsize=(8, 4))
for cls_id, cls_name in CLASS_NAMES.items():
    sub = df[df["class_id"] == cls_id]
    ax.hist(sub["aspect_ratio"], bins=50, alpha=0.6, label=cls_name)
ax.set_xlabel("Aspect Ratio (w/h)")
ax.set_ylabel("Count")
ax.set_title("Bbox Aspect Ratio Distribution")
ax.legend()
plt.tight_layout()
plt.show()

## 3. 图像属性

In [ ]:
# Sample image resolutions and file sizes
img_dir = DATA_ROOT / "train" / "images"
all_imgs = sorted(img_dir.glob("*"))[:500]

resolutions = []
file_sizes = []
for p in all_imgs:
    img = cv2.imread(str(p))
    if img is not None:
        h, w = img.shape[:2]
        resolutions.append((w, h))
        file_sizes.append(p.stat().st_size / 1024)  # KB

res_df = pd.DataFrame(resolutions, columns=["width", "height"])
print("Resolution stats:")
print(res_df.describe())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.hist(res_df["width"], bins=30, alpha=0.7, label="width")
ax1.hist(res_df["height"], bins=30, alpha=0.7, label="height")
ax1.set_xlabel("Pixels")
ax1.set_title("Image Resolution Distribution")
ax1.legend()

ax2.hist(file_sizes, bins=30, color="#54A24B", alpha=0.8)
ax2.set_xlabel("File Size (KB)")
ax2.set_title("Image File Size Distribution")
plt.tight_layout()
plt.show()

## 4. 数据质量摘要

Run the validation script for a complete quality report:

```bash
python training/validate_dataset.py --data-root datasets/RPS --output report.json
```